# Data transcript analysis

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Data_used.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title Load Git Repository
# @markdown Clone the Git repository for analysis if needed.
clone_git_repo = False  # @param {type:"boolean"}
if clone_git_repo:
    !git clone https://github.com/PeaceAndLongLife/Analysis-Colab.git
%cd Analysis-Colab

import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))
%cd notebooks


/content/Analysis-Colab
/content/Analysis-Colab/notebooks


## Setup Google API

In [6]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FIL
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')

# @title ## Install pyDrive  {"form-width":"20%"}

# @markdown ---
# @markdown Installing PyDrive
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
# sys.path.append(PACKAGE_PATH)
sys.path.append('src')
# 3. Import your file!
from GoogleFunctions import extract_file_id
from local_io import read_csv_from_id


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The 'extract_file_id' function has been defined.


## Read in data


In [33]:
# @title Data file info {"form-width":"20%"}

# UserProfile Info
# @markdown Read in userprofile file
userprofile_file_link = "https://drive.google.com/file/d/1zJW3xoQx7sworuVDZRgsSHsZta-M6Xt9/view?usp=drive_link" # @param {"type":"string"}
userprofile_file_link_id = extract_file_id(userprofile_file_link)
show_userprofile = False # @param {"type":"boolean"}

# Consent data info
# @markdown Read in consent file

consent_file_link = "https://drive.google.com/file/d/1shPdP-UQnDISurjIWQNPTtiYDGzn0Erm/view?usp=drive_link" # @param {"type":"string"}
consent_file_link_id = extract_file_id(consent_file_link)
show_consent = False # @param {"type":"boolean"}

# Transcript Data info
# @markdown Read in transcript file
trans_file_link = "https://drive.google.com/file/d/1AxlUwk6ucxVjhRkuQv0tgDvTFlDkipsu/view?usp=drive_link" # @param {"type":"string"}
trans_file_link_id = extract_file_id(trans_file_link)
show_trans = False # @param {"type":"boolean"}
show_remove_staff_data = False # @param {"type":"boolean"}

userprofile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, userprofile_file_link_id, show_userprofile)
consent_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, consent_file_link_id, show_consent)
trans_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans_file_link_id, show_trans)

from data_scrub import remove_staff, process_messages, parse_messages
# import json

if show_remove_staff_data:
    userprofile_df, trans_df = remove_staff(
        userprofile_df,
        trans_df,
        [
          'travis@pdx.edu',
          'marie.snetinova@matfyz.cuni.cz',
          'mptl_group_1@pdx.edu',
          'mptl_group_2@pdx.edu',
          'mptl_group_3@pdx.edu',
          'mptl_group_4@pdx.edu',
          'mptl_group_5@pdx.edu',
          'mptl_group_6@pdx.edu',
          'mptl_group_1@pdx.edu',
          'mptl_user_1@pdx.edu',
          'mptl_user_10@pdx.edu'
        ])

# Merge consent into transcript
combined_df = process_messages(consent_df, trans_df)


# Apply the parsing function to the 'all_messages' column
combined_df['all_messages'] = combined_df['all_messages'].apply(parse_messages)

# Count message objects in 'all_messages' and add to 'interactions' column
combined_df['interactions'] = combined_df['all_messages'].apply(len)

combined_df = combined_df[combined_df['all_messages'].apply(lambda x: len(x) > 0)]

print("Combined DataFrame with split question columns and interactions:")
# display(combined_df.head())

Google Drive API service built successfully.
Reading in csv file. id=1zJW3xoQx7sworuVDZRgsSHsZta-M6Xt9
Google Drive API service built successfully.
Reading in csv file. id=1shPdP-UQnDISurjIWQNPTtiYDGzn0Erm
Google Drive API service built successfully.
Reading in csv file. id=1AxlUwk6ucxVjhRkuQv0tgDvTFlDkipsu
Consent Data Combined into DataFrame.
 Question Data parsed and updated.
Combined DataFrame with split question columns and interactions:


In [34]:
from data_scrub import explode_json_messages
combined_df = explode_json_messages(combined_df)


In [35]:
# @title analyze_stats function
def analyze_hist(df, drop_counts_under_param,y,x):
  # Exclude users with less than drop_counts_under_param unique threads
  df_filtered = df[df[y] >= drop_counts_under_param]


  ### Plot distribution
  import matplotlib.pyplot as plt
  import seaborn as sns

  # Set the style for the plot
  sns.set_style("whitegrid")

  plt.figure(figsize=(10, 6))
  sns.histplot(df_filtered[y], bins=5, kde=True)
  plt.title(f"Distribution of {y} per {x}")
  plt.xlabel(f"{x}")
  # plt.ylabel(f"Number of {y}")
  plt.tight_layout()
  plt.show()
  display(df_filtered[y].describe())


# Report

In [59]:
# @title Report user ratings
import ipywidgets as widgets
from IPython.display import display, clear_output

data_df = combined_df[combined_df['data']==True]

# Calculate rating counts per user to sort the dropdown options
data_user_counts = data_df['user'].value_counts().sort_values(ascending=False)
sorted_users = data_user_counts.index.tolist()

# Add an 'All Users' option to the beginning of the list
sorted_users.insert(0, 'All Users')

# Create the dropdown widget
user_dropdown = widgets.Dropdown(
    options=sorted_users,
    value='All Users',
    description='Select User:',
    disabled=False,
)
# Create an output widget to display the filtered DataFrame
output = widgets.Output()
global filtered_df

def on_user_change(change):
    with output:
        clear_output(wait=True)
        selected_user = change.new

        if selected_user == 'All Users':
            display(data_df)
        else:
            filtered_df = data_df[data_df['user'] == selected_user]

        user_rating_report = filtered_df[['user', 'Course', 'lab_number', 'question_number', 'data']].sort_values(by=['user', 'Course', 'lab_number', 'question_number']).reset_index(drop=True)
        display(user_rating_report)

# Observe changes in the dropdown value
user_dropdown.observe(on_user_change, names='value')




In [60]:
# @title Display Output
# Display the dropdown and the initial output
display(user_dropdown, output)

# Initial display of the full DataFrame when 'All Users' is selected
with output:
  user_rating_report = data_df[['user', 'Course', 'lab_number', 'question_number']].sort_values(by=['user', 'Course', 'lab_number', 'question_number']).reset_index(drop=True)
  display(user_rating_report)

Dropdown(description='Select User:', options=('All Users', 163, 139, 203, 149, 225, 219, 106, 249, 215, 98, 25…

Output()

In [61]:
# @title data Stats
data_df = combined_df[combined_df['data']==True]
# analyze_hist(data_df, 0,'data','user')

print("\n---\n")
print('Total data counts')
display(data_df['data'].value_counts().sort_index())

user_data_counts = data_df.groupby(['user', 'Course','lab_number', 'question_number'])['data'].value_counts().unstack(fill_value=0)
display(user_data_counts[user_data_counts[True]>0].sort_values(by=True, ascending=False))


---

Total data counts


,count
data,
True,355


data                                            True
user Course         lab_number question_number      
163  PH214 (online) 05         00                 30
                    08         00                 14
139  PH215 (online) 02         03                 12
203  PH214 (online) 07         11                 10
                    06         08                 10
...                                              ...
249  PH214 (online) 01         00                  1
225  PH214 (online) 03         03                  1
249  PH214 (online) 01         10                  1
225  PH214 (online) 03         04                  1
234  PH214 (online) 02         02                  1

[128 rows x 1 columns]